In [ ]:
p

In [ ]:
print("="*120)
print("FINAL RECOMMENDATIONS")
print("="*120)

# Count by risk
high_risk = df_analysis[df_analysis['risk'].str.contains('HIGH', na=False)]
production = df_analysis[df_analysis['risk'] == 'NONE']
review_needed = df_analysis[df_analysis['risk'] == 'INVESTIGATE']

print(f"\nâœ… PRODUCTION TABLES (KEEP): {len(production)}")
for _, row in production.iterrows():
    rows_str = f"{row['row_count']:,}" if row['row_count'] is not None else "Unknown"
    print(f"   - {row['table']:50s} ({rows_str} rows)")

if len(high_risk) > 0:
    print(f"\nðŸ—‘ï¸  HIGH PRIORITY CLEANUP: {len(high_risk)} tables")
    for _, row in high_risk.iterrows():
        rows_str = f"{row['row_count']:,}" if row['row_count'] is not None else "Unknown"
        print(f"   - {row['table']:50s} ({rows_str} rows) - {row['justification']}")

if len(review_needed) > 0:
    print(f"\nâ“ REVIEW NEEDED: {len(review_needed)} tables")
    for _, row in review_needed.iterrows():
        rows_str = f"{row['row_count']:,}" if row['row_count'] is not None else "Unknown"
        print(f"   - {row['table']:50s} ({rows_str} rows)")

print("\n" + "="*120)
print("NEXT STEPS:")
print("="*120)
print("1. Review tables marked 'INVESTIGATE' to confirm they're not needed")
print("2. Backup any tables before dropping (if needed)")
print("3. Run the cleanup SQL script generated below")
print("4. Verify Phase 1-3 notebooks still run after cleanup")
print("\nEstimated cleanup benefit:")
total_rows_to_drop = high_risk['row_count'].sum() if len(high_risk) > 0 else 0
print(f"  - Remove {len(high_risk)} unused tables")
print(f"  - Free up ~{total_rows_to_drop:,} rows of storage")
print(f"  - Simplify gold schema for maintenance")

## 6. Recommendations Summary

**Action Items**:

In [ ]:
# Generate cleanup SQL script
cleanup_sql = []
cleanup_sql.append("-- ================================================================")
cleanup_sql.append("-- Gold Schema Cleanup Script")
cleanup_sql.append(f"-- Generated: {pd.Timestamp.now()}")
cleanup_sql.append(f"-- Tables to drop: {len(safe_to_drop)}")
cleanup_sql.append("-- ================================================================")
cleanup_sql.append("")
cleanup_sql.append("-- IMPORTANT: Review each table before dropping!")
cleanup_sql.append("-- Test in dev environment first")
cleanup_sql.append("")

if len(safe_to_drop) > 0:
    cleanup_sql.append("-- ================================================================")
    cleanup_sql.append("-- HIGH PRIORITY: Orphaned/Unused Tables")
    cleanup_sql.append("-- ================================================================")
    cleanup_sql.append("")
    
    for _, row in safe_to_drop.iterrows():
        table = row['table']
        status = row['status']
        
        cleanup_sql.append(f"-- Table: {table}")
        cleanup_sql.append(f"-- Status: {status}")
        cleanup_sql.append(f"-- Created by: {row['created_by']}")
        cleanup_sql.append(f"-- Read by: {row['read_by']}")
        cleanup_sql.append("")
        cleanup_sql.append(f"-- Backup (optional):")
        cleanup_sql.append(f"-- CREATE TABLE {table}_backup AS SELECT * FROM {table};")
        cleanup_sql.append("")
        cleanup_sql.append(f"DROP TABLE IF EXISTS {table};")
        cleanup_sql.append("")
        cleanup_sql.append("-- ----------------------------------------------------------------")
        cleanup_sql.append("")
    
    # Save to file
    script_path = os.path.join(os.path.dirname(os.getcwd()), "gold_schema_cleanup.sql")
    with open(script_path, 'w') as f:
        f.write('\n'.join(cleanup_sql))
    
    print(f"âœ… Cleanup script saved to: {script_path}")
    print("\n" + "="*80)
    print("PREVIEW OF CLEANUP SCRIPT:")
    print("="*80)
    print('\n'.join(cleanup_sql[:50]))  # Show first 50 lines
else:
    print("âœ… No unused tables found - schema is clean!")

## 5. Generate Cleanup SQL Script

Create SQL statements to drop unused tables (with safety checks):

In [ ]:
# Filter tables safe to drop
safe_to_drop = df_analysis[
    df_analysis['risk'].str.contains('HIGH', na=False)
].sort_values('row_count', ascending=True)

print("=" * 120)
print(f"ðŸ—‘ï¸  SAFE TO DROP: {len(safe_to_drop)} TABLES")
print("=" * 120)
if len(safe_to_drop) > 0:
    print(safe_to_drop[['table', 'row_count', 'status', 'risk', 'justification']].to_string(index=False))
else:
    print("No tables identified for cleanup")

print("\n\n" + "=" * 120)
print("ðŸ“Š SUMMARY BY RISK CATEGORY:")
print("=" * 120)
risk_summary = df_analysis.groupby('risk').agg({
    'table': 'count',
    'row_count': 'sum'
}).rename(columns={'table': 'count', 'row_count': 'total_rows'})
print(risk_summary.to_string())

print("\n\n" + "=" * 120)
print("ðŸ“‹ PRODUCTION TABLES (KEEP):")
print("=" * 120)
production_tables = df_analysis[df_analysis['risk'] == 'NONE']
for _, row in production_tables.iterrows():
    print(f"  {row['table']:50s} {row['row_count']:>15,} rows  - {row['justification']}")

## 4. Generate Safe-to-Drop Recommendations

Prioritize tables by cleanup risk and potential value:

In [ ]:
# Categorize tables based on known production pipeline
all_tables = [f'gold.{t}' for t in tables_list]

usage_analysis = []

for table in sorted(all_tables):
    # Determine status based on production pipeline knowledge
    if table in PRODUCTION_TABLES:
        status = "âœ… PRODUCTION (KEEP)"
        risk = "NONE"
        justification = "Core production pipeline table"
    elif table in SUPERSEDED_TABLES:
        status = "âš ï¸  SUPERSEDED"
        risk = "HIGH - can drop"
        justification = "Replaced by newer version"
    elif table.startswith('gold.gads_'):
        # GADS source tables
        status = "âœ… GADS SOURCE (KEEP)"
        risk = "NONE"
        justification = "Source data from GADS ingestion"
    elif 'training_dataset' in table and 'v2' not in table:
        status = "âš ï¸  OLD VERSION"
        risk = "HIGH - superseded"
        justification = "Older version of training dataset"
    elif any(keyword in table for keyword in ['temp', 'test', 'scratch', 'backup']):
        status = "âŒ TEMPORARY"
        risk = "HIGH - orphaned"
        justification = "Temporary/test table"
    else:
        status = "â“ UNKNOWN - REVIEW NEEDED"
        risk = "INVESTIGATE"
        justification = "Not in documented pipeline"
    
    # Get row count from metadata
    row_count = next((m['row_count'] for m in table_metadata if m['table'] == table), None)
    
    usage_analysis.append({
        'table': table,
        'row_count': row_count,
        'status': status,
        'risk': risk,
        'justification': justification
    })

df_analysis = pd.DataFrame(usage_analysis)

print("=" * 120)
print("GOLD SCHEMA TABLE ANALYSIS")
print("=" * 120)
print(df_analysis.to_string(index=False))
print("\n" + "=" * 120)

## 3. Analyze Table Usage Patterns

Cross-reference created vs read to categorize tables:

In [ ]:
# Production pipeline tables (KEEP - actively used)
PRODUCTION_TABLES = {
    # Phase 1 outputs
    'gold.predictive_labels',           # Label construction from GADS
    'gold.running_indicator',           # Asset running thresholds
    
    # Phase 2 outputs (v2 - current)
    'gold.training_dataset_v2',         # Final training dataset (Phase 3 input)
    'gold.sensor_correlation_ranking',  # Time-lagged correlations
    'gold.phase2_selected_tags',        # Top sensors per asset
    
    # Supporting reference tables
    'gold.fact_pi',                     # PI Historian data
    'gold.bridge_pi_tag_to_asset',     # PI tag metadata
    'gold.fact_icare_measurement',     # iCare sensor data
    'gold.dim_equipment',               # Equipment hierarchy
    
    # GADS source tables (inputs)
    # Note: You have ~20 GADS tables, list specific ones if needed
}

# Superseded tables (candidates for cleanup)
SUPERSEDED_TABLES = {
    'gold.training_dataset',   # v1 - replaced by v2
    # Add others as discovered
}

print("âœ… Production pipeline has", len(PRODUCTION_TABLES), "core tables")
print("\nâš ï¸  Potential cleanup candidates will be identified...")

## 2. Production Pipeline Tables (Known Active)

Define the tables we **know** are active in the production pipeline based on your memory:

## 1. Query Gold Schema Tables Directly

Use Spark SQL to discover all tables in the gold schema and analyze their metadata:

In [ ]:
# Query all tables in gold schema
tables_df = spark.sql("SHOW TABLES IN gold")
tables_list = [row.tableName for row in tables_df.collect()]

print(f"Found {len(tables_list)} tables in gold schema:")
for table in sorted(tables_list):
    print(f"  - gold.{table}")

# Get table metadata (row counts, size estimates)
table_metadata = []
for table in tables_list:
    try:
        count = spark.table(f"gold.{table}").count()
        table_metadata.append({
            'table': f'gold.{table}',
            'row_count': count
        })
    except Exception as e:
        table_metadata.append({
            'table': f'gold.{table}',
            'row_count': None,
            'error': str(e)
        })

print("\nâœ“ Table metadata collected")

In [ ]:
# Import required libraries
import os
import json
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

print("âœ“ Libraries imported")
print(f"Current directory: {os.getcwd()}")

# Gold Schema Cleanup Analysis

**Objective**: Analyze the gold schema to identify unused tables that can be safely dropped.

## Approach:
1. Query `SHOW TABLES IN gold` to get all existing tables
2. Categorize tables based on documented production pipeline (Phase 1-3)
3. Identify superseded versions and temporary tables
4. Generate cleanup recommendations and SQL scripts

**Production Pipeline** (Phase 1-3):
- Phase 1: `gold.predictive_labels`, `gold.running_indicator`
- Phase 2: `gold.training_dataset_v2`, `gold.sensor_correlation_ranking`, `gold.phase2_selected_tags`
- Phase 3: Uses Phase 2 training dataset only
- Supporting: `gold.fact_pi`, `gold.bridge_pi_tag_to_asset`, `gold.fact_icare_measurement`, `gold.dim_equipment`
- Source: ~20 GADS tables (from Process-GADS-Data.ipynb)

**Expected to find**: `gold.training_dataset` (v1 - superseded by v2)